# Training at scale with the Vertex AI Training Service
**Learning Objectives:**
  1. Learn how to organize your training code into a Python package
  1. Train your model using cloud infrastructure via Google Cloud Vertex AI Training Service
  1. Learn how to run your training package using Docker containers and push training Docker images on a Docker registry
  1. Monitor Cloud job using TensorBoard

## Introduction

In this notebook we'll make the jump from training locally, to training in the cloud. We'll take advantage of Google Cloud's [Vertex AI Training Service](https://cloud.google.com/vertex-ai/). 

Vertex AI Training Service is a managed service that allows the training and deployment of ML models without having to provision or maintain servers. The infrastructure is handled seamlessly by the managed service for us.

Each learning objective will correspond to a __#TODO__ in the [student lab notebook](../labs/1_training_at_scale_vertex.ipynb) -- try to complete that notebook first before reviewing this solution notebook.

Specify your project name and bucket name in the cell below.

In [1]:
import os
import warnings
from datetime import datetime

import tensorboard
from google import api_core
from google.cloud import aiplatform, bigquery


%load_ext tensorboard
%load_ext google.cloud.bigquery
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

/home/jupyter/asl-ml-immersion/asl_core/.venv/lib/python3.12/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


The google.cloud.bigquery extension is already loaded. To reload it, use:
  %reload_ext google.cloud.bigquery


Change the following cell as necessary:

In [2]:
# Change below if necessary
PROJECT = !gcloud config get-value project  # noqa: E999
PROJECT = PROJECT[0]
BUCKET = PROJECT
REGION = "us-central1"

OUTDIR = f"gs://{BUCKET}/taxifare/data"

%env PROJECT=$PROJECT
%env BUCKET=$BUCKET
%env REGION=$REGION
%env OUTDIR=$OUTDIR

env: PROJECT=qwiklabs-asl-00-8534302ca246
env: BUCKET=qwiklabs-asl-00-8534302ca246
env: REGION=us-central1
env: OUTDIR=gs://qwiklabs-asl-00-8534302ca246/taxifare/data


Confirm below that the bucket is regional and its region equals to the specified region:

In [3]:
%%bash
gsutil ls -Lb gs://$BUCKET | grep "gs://\|Location"
echo $REGION

gs://qwiklabs-asl-00-8534302ca246/ :
	Location type:			region
	Location constraint:		US-CENTRAL1
us-central1


In [4]:
%%bash
gcloud config set project $PROJECT
gcloud config set ai/region $REGION

Project 'qwiklabs-asl-00-8534302ca246' lacks an 'environment' tag. Please create or add a tag with key 'environment' and a value like 'Production', 'Development', 'Test', or 'Staging'. Add an 'environment' tag using `gcloud resource-manager tags bindings create`. See https://cloud.google.com/resource-manager/docs/creating-managing-projects#designate_project_environments_with_tags for details.
Updated property [core/project].
Updated property [ai/region].


## Create BigQuery tables

If you have not already created a BigQuery dataset for our data, run the following cell:

In [5]:
bq = bigquery.Client(project=PROJECT)
dataset = bigquery.Dataset(bq.dataset("taxifare"))

try:
    bq.create_dataset(dataset)
    print("Dataset created")
except api_core.exceptions.Conflict:
    print("Dataset already exists")

Dataset created


Let's create a table with 1 million examples.

This query reflects the best practice of using a hash function (`FARM_FINGERPRINT`) in the `WHERE` and `ORDER BY` clauses to ensure reproducibility while introducing randomness.

Note that the order of columns is exactly what was in our CSV files.

In [6]:
%%bigquery

CREATE OR REPLACE TABLE taxifare.feateng_training_data AS

SELECT
    (tolls_amount + fare_amount) AS fare_amount,
    pickup_datetime,
    pickup_longitude AS pickuplon,
    pickup_latitude AS pickuplat,
    dropoff_longitude AS dropofflon,
    dropoff_latitude AS dropofflat,
    passenger_count*1.0 AS passengers,
    'unused' AS key
FROM `nyc-tlc.yellow.trips` as ny_taxi_trips
WHERE ABS(MOD(FARM_FINGERPRINT(CAST(pickup_datetime AS STRING)), 1000)) = 1
AND
    trip_distance > 0
    AND fare_amount >= 2.5
    AND pickup_longitude > -78
    AND pickup_longitude < -70
    AND dropoff_longitude > -78
    AND dropoff_longitude < -70
    AND pickup_latitude > 37
    AND pickup_latitude < 45
    AND dropoff_latitude > 37
    AND dropoff_latitude < 45
    AND passenger_count > 0
ORDER BY FARM_FINGERPRINT(TO_JSON_STRING(ny_taxi_trips))

Query is running:   0%|          |

""


Make the validation dataset be 1/10 the size of the training dataset.

In [7]:
%%bigquery

CREATE OR REPLACE TABLE taxifare.feateng_valid_data AS

SELECT
    (tolls_amount + fare_amount) AS fare_amount,
    pickup_datetime,
    pickup_longitude AS pickuplon,
    pickup_latitude AS pickuplat,
    dropoff_longitude AS dropofflon,
    dropoff_latitude AS dropofflat,
    passenger_count*1.0 AS passengers,
    'unused' AS key
FROM `nyc-tlc.yellow.trips` as ny_taxi_trips
WHERE ABS(MOD(FARM_FINGERPRINT(CAST(pickup_datetime AS STRING)), 10000)) = 2
AND
    trip_distance > 0
    AND fare_amount >= 2.5
    AND pickup_longitude > -78
    AND pickup_longitude < -70
    AND dropoff_longitude > -78
    AND dropoff_longitude < -70
    AND pickup_latitude > 37
    AND pickup_latitude < 45
    AND dropoff_latitude > 37
    AND dropoff_latitude < 45
    AND passenger_count > 0
ORDER BY FARM_FINGERPRINT(TO_JSON_STRING(ny_taxi_trips))

Query is running:   0%|          |

""


## Export the tables as CSV files

In [8]:
%%bash

echo "Deleting current contents of $OUTDIR"
gsutil -m -q rm -rf $OUTDIR

echo "Extracting training data to $OUTDIR"
bq --location=US extract \
   --destination_format CSV  \
   --field_delimiter "," --noprint_header \
   taxifare.feateng_training_data \
   $OUTDIR/taxi-train-*.csv

echo "Extracting validation data to $OUTDIR"
bq --location=US extract \
   --destination_format CSV  \
   --field_delimiter "," --noprint_header \
   taxifare.feateng_valid_data \
   $OUTDIR/taxi-valid-*.csv

gsutil ls -l $OUTDIR

Deleting current contents of gs://qwiklabs-asl-00-8534302ca246/taxifare/data


CommandException: 1 files/objects could not be removed.


Extracting training data to gs://qwiklabs-asl-00-8534302ca246/taxifare/data


Waiting on bqjob_r79aa52ce320dd323_0000019c9653e588_1 ... (10s) Current status: DONE   


Extracting validation data to gs://qwiklabs-asl-00-8534302ca246/taxifare/data


Waiting on bqjob_r38084c65543d95f5_0000019c96541d82_1 ... (2s) Current status: DONE   


  88345235  2026-02-25T19:43:37Z  gs://qwiklabs-asl-00-8534302ca246/taxifare/data/taxi-train-000000000000.csv
   8725746  2026-02-25T19:43:43Z  gs://qwiklabs-asl-00-8534302ca246/taxifare/data/taxi-valid-000000000000.csv
TOTAL: 2 objects, 97070981 bytes (92.57 MiB)


Confirm that you have created both the training and validation datasets in Google Cloud Storage.

In [9]:
!gsutil ls gs://$BUCKET/taxifare/data

gs://qwiklabs-asl-00-8534302ca246/taxifare/data/taxi-train-000000000000.csv
gs://qwiklabs-asl-00-8534302ca246/taxifare/data/taxi-valid-000000000000.csv


In [10]:
!gsutil cat gs://$BUCKET/taxifare/data/taxi-train-000000000000.csv | head -2

13.3,2010-01-19 18:43:00 UTC,-74.015138,40.714373,-73.981883,40.763583,1,unused
6.9,2009-11-27 10:28:00 UTC,-73.97902,40.762145,-73.978778,40.76196,3,unused


In [11]:
!gsutil cat gs://$BUCKET/taxifare/data/taxi-valid-000000000000.csv | head -2

5.7,2011-07-12 08:41:23 UTC,-73.991075,40.742206,-73.991361,40.751948,1,unused
11.5,2015-03-04 14:53:42 UTC,-73.961074829101562,40.76715087890625,-73.981941223144531,40.779548645019531,1,unused


## Make code compatible with Vertex AI Training Service
In order to make our code compatible with Vertex AI Training Service we need to make the following changes:

1. Upload data to Google Cloud Storage 
2. Move code into a trainer Python package
4. Submit training job with `gcloud` to train on Vertex AI

### Move code into a python package

The first thing to do is to convert your training code snippets into a regular Python package. 

A Python package is simply a collection of one or more `.py` files along with an `__init__.py` file to identify the containing directory as a package. The `__init__.py` sometimes contains initialization code but for our purposes an empty file is sufficient.

#### Create the package directory

Our package directory contains 3 files:

In [12]:
ls ./taxifare/trainer/

__init__.py  model.py  task.py


#### Paste existing code into model.py

A Python package requires our code to be in a .py file, as opposed to notebook cells. So, we simply copy and paste our existing code we 
developed in [this notebook](../../introduction_to_tensorflow/solutions/5_custom_feature_engineering.ipynb) into a single file.

**Lab Task #1**: Organizing your training code into a Python package

There are two places to fill in TODOs in `model.py`. 

 * **TODO 1a**: in the `build_dnn_model` function, add code to compile the model using an optimizer with a custom learning rate.
 * **TODO 1b**: in the `train_and_evaluate` function, add code to define variables using the `hparams` dictionary.

In [14]:
%%writefile ./taxifare/trainer/model.py
"""Data prep, train and evaluate DNN model."""

import logging
import os

import numpy as np
import tensorflow as tf
import keras
from keras import callbacks
from keras.layers import (
    Concatenate,
    Dense,
    Discretization,
    Embedding,
    Flatten,
    HashedCrossing,
    Input,
    Lambda,
)
from keras.metrics import RootMeanSquaredError

def parse_csv(row):
    ds = tf.strings.split(row, ",")
    # Label: fare_amount
    label = tf.strings.to_number(ds[0])
    # Features: pickup_longitude, pickup_latitude, dropoff_longitude, dropoff_latitude
    feature = tf.strings.to_number(ds[2:6])  # use some features only
    # Passing feature in tuple so that we can handle them separately.
    return (feature[0], feature[1], feature[2], feature[3]), label


def create_dataset(pattern, batch_size, num_repeat, mode="eval"):
    ds = tf.data.Dataset.list_files(pattern)
    ds = ds.flat_map(tf.data.TextLineDataset)
    ds = ds.map(parse_csv)
    if mode == "train":
        ds = ds.shuffle(buffer_size=1000)
    ds = ds.repeat(num_repeat).batch(batch_size, drop_remainder=True)
    return ds

def parse_lat_lon(row):
    columns = tf.strings.split(row, ",")
    # latitude idx: 3 and 5, longitude idx: 2 and 4
    lat_strings = tf.gather(columns, [3, 5])
    lon_strings = tf.gather(columns, [2, 4])
    lat_features = tf.strings.to_number(lat_strings)
    lon_features = tf.strings.to_number(lon_strings)
    return lat_features, lon_features

def adapt_normalize(train_data_path):
    ds = tf.data.Dataset.list_files(train_data_path)
    ds = ds.flat_map(tf.data.TextLineDataset)
    lat_lon_ds = ds.map(parse_lat_lon).batch(10000)

    lat_scaler = keras.layers.Normalization(axis=None)
    lon_scaler = keras.layers.Normalization(axis=None)

    lat_values, lon_values = next(iter(lat_lon_ds))

    lat_scaler.adapt(lat_values)
    lon_scaler.adapt(lon_values)

    print("Computed statistics for latitude:")
    print(f"mean: {lat_scaler.mean}, variance: {lat_scaler.variance}")
    print("+++++")
    print("Computed statistics for longitude:")
    print(f"mean: {lon_scaler.mean}, variance: {lon_scaler.variance}")

    return lat_scaler, lon_scaler


def euclidean(params):
    lon1, lat1, lon2, lat2 = params
    londiff = lon2 - lon1
    latdiff = lat2 - lat1
    return tf.sqrt(londiff * londiff + latdiff * latdiff)


def transform(inputs, nbuckets, normalizers):
    lat_scaler, lon_scaler = normalizers

    # Normalize longitude
    scaled_plon = lon_scaler(inputs["pickup_longitude"])
    scaled_dlon = lon_scaler(inputs["dropoff_longitude"])

    # Normalize latitude
    scaled_plat = lat_scaler(inputs["pickup_latitude"])
    scaled_dlat = lat_scaler(inputs["dropoff_latitude"])

    # Lambda layer for the custom euclidean function
    euclidean_distance = Lambda(euclidean, name="euclidean")(
        [scaled_plon, scaled_plat, scaled_dlon, scaled_dlat]
    )

    # Discretization
    latbuckets = np.linspace(start=-5, stop=5, num=nbuckets).tolist()
    lonbuckets = np.linspace(start=-5, stop=5, num=nbuckets).tolist()

    plon = Discretization(lonbuckets, name="plon_bkt")(scaled_plon)
    plat = Discretization(latbuckets, name="plat_bkt")(scaled_plat)
    dlon = Discretization(lonbuckets, name="dlon_bkt")(scaled_dlon)
    dlat = Discretization(latbuckets, name="dlat_bkt")(scaled_dlat)

    # Feature Cross with HashedCrossing layer
    p_fc = HashedCrossing(num_bins=(nbuckets + 1) ** 2, name="p_fc")((plon, plat))
    d_fc = HashedCrossing(num_bins=(nbuckets + 1) ** 2, name="d_fc")((dlon, dlat))
    pd_fc = HashedCrossing(num_bins=(nbuckets + 1) ** 4, name="pd_fc")((p_fc, d_fc))

    # Embedding with Embedding layer
    pd_embed = Flatten()(
        Embedding(input_dim=(nbuckets + 1) ** 4, output_dim=10, name="pd_embed")(
            pd_fc
        )
    )

    transformed = Concatenate()([
        scaled_plon,
        scaled_dlon,
        scaled_plat,
        scaled_dlat,
        euclidean_distance, 
        pd_embed
    ])

    return transformed


def build_dnn_model(nbuckets, nnsize, lr, normalizers):
    INPUT_COLS = [
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude",
    ]

    inputs = {
        colname: Input(name=colname, shape=(1,), dtype="float32")
        for colname in INPUT_COLS
    }

    # transforms
    x = transform(inputs, nbuckets, normalizers)

    for layer, nodes in enumerate(nnsize):
        x = Dense(nodes, activation="relu", name=f"h{layer}")(x)
    output = Dense(1, name="fare")(x)

    model = keras.Model(inputs=list(inputs.values()), outputs=output)

    lr_optimizer = keras.optimizers.Adam(learning_rate=lr)
    model.compile(optimizer=lr_optimizer, loss="mse", metrics=[RootMeanSquaredError()])


    return model


def train_and_evaluate(hparams):
    # TODO 1b: Your code here
    batch_size = hparams["batch_size"]
    nbuckets = hparams["nbuckets"]
    lr = hparams["lr"]
    nnsize = [int(s) for s in hparams["nnsize"].split()]
    eval_data_path = hparams["eval_data_path"]
    num_evals = hparams["num_evals"]
    num_examples_to_train_on = hparams["num_examples_to_train_on"]
    output_dir = hparams["output_dir"]
    train_data_path = hparams["train_data_path"]

    model_export_path = os.path.join(output_dir, "model.keras")
    serving_model_export_path = os.path.join(output_dir, "savedmodel")
    checkpoint_path = os.path.join(output_dir, "checkpoint.keras")
    tensorboard_path = os.path.join(output_dir, "tensorboard")

    if tf.io.gfile.exists(output_dir):
        tf.io.gfile.rmtree(output_dir)

    normalizers = adapt_normalize(eval_data_path)

    model = build_dnn_model(nbuckets, nnsize, lr, normalizers)
    logging.info(model.summary())
    
    trainds = create_dataset(
        pattern=train_data_path, batch_size=batch_size, num_repeat=None, mode="train"
    )

    evalds = create_dataset(
        pattern=eval_data_path, batch_size=batch_size, num_repeat=1, mode="eval"
    )

    steps_per_epoch = num_examples_to_train_on // (batch_size * num_evals)

    checkpoint_cb = callbacks.ModelCheckpoint(
        checkpoint_path, verbose=1
    )
    tensorboard_cb = callbacks.TensorBoard(tensorboard_path, histogram_freq=1)

    history = model.fit(
        trainds,
        validation_data=evalds,
        epochs=num_evals,
        steps_per_epoch=max(1, steps_per_epoch),
        verbose=2,  # 0=silent, 1=progress bar, 2=one line per epoch
        callbacks=[checkpoint_cb, tensorboard_cb],
    )

    # Save the Keras model file.
    model.save(model_export_path)
    # Exporting the model in savedmodel for serving.
    model.export(serving_model_export_path)
    return history

Overwriting ./taxifare/trainer/model.py


### Define Command Line Parser

If you look closely above, you'll notice a new function, `train_and_evaluate` that wraps the code that actually trains the model. This allows us to parametrize the training by passing a dictionary of parameters to this function (e.g, `batch_size`, `num_examples_to_train_on`, `train_data_path` etc.)

This is useful because the output directory, data paths and number of train steps will be different depending on whether we're training locally or in the cloud. Parametrizing allows us to use the same code for both.

We specify these parameters at run time via the command line. Which means we need to add code to parse command line parameters and invoke `train_and_evaluate()` with those params. This is the job of the `task.py` file. 

In [15]:
%%writefile taxifare/trainer/task.py
"""Argument definitions for model training code in `trainer.model`."""

import argparse

from trainer import model

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--batch_size",
        help="Batch size for training steps",
        type=int,
        default=32,
    )
    parser.add_argument(
        "--eval_data_path",
        help="GCS location pattern of eval files",
        required=True,
    )
    parser.add_argument(
        "--nnsize",
        help="Hidden layer sizes (provide space-separated sizes)",
        default="32 8",
    )
    parser.add_argument(
        "--nbuckets",
        help="Number of buckets to divide lat and lon with",
        type=int,
        default=10,
    )
    parser.add_argument(
        "--lr", help="learning rate for optimizer", type=float, default=0.001
    )
    parser.add_argument(
        "--num_evals",
        help="Number of times to evaluate model on eval data training.",
        type=int,
        default=5,
    )
    parser.add_argument(
        "--num_examples_to_train_on",
        help="Number of examples to train on.",
        type=int,
        default=100,
    )
    parser.add_argument(
        "--output_dir",
        help="GCS location to write checkpoints and export models",
        required=True,
    )
    parser.add_argument(
        "--train_data_path",
        help="GCS location pattern of train files containing eval URLs",
        required=True,
    )
    args = parser.parse_args()
    hparams = args.__dict__

    model.train_and_evaluate(hparams)


Overwriting taxifare/trainer/task.py


### Run trainer module package locally

Now we can test our training code locally as follows using the local test data. We'll run a very small training job over a single file with a small batch size and one eval step.

In [16]:
%%bash

EVAL_DATA_PATH=../data/taxi-traffic-valid*
TRAIN_DATA_PATH=../data/taxi-traffic-train*
OUTPUT_DIR=./taxifare-model

test ${OUTPUT_DIR} && rm -rf ${OUTPUT_DIR}
export PYTHONPATH=${PYTHONPATH}:${PWD}/taxifare

python3 -m trainer.task \
--eval_data_path $EVAL_DATA_PATH \
--output_dir $OUTPUT_DIR \
--train_data_path $TRAIN_DATA_PATH \
--batch_size 5 \
--num_examples_to_train_on 100 \
--num_evals 1 \
--nbuckets 10 \
--lr 0.001 \
--nnsize "32 8"

2026-02-25 19:51:46.493142: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772049106.782249   12393 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772049106.864449   12393 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-25 19:51:56.987599: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Computed statistics for latitude:
mean: [[-73.9746]], variance: [[0.00169989]]
+++++
Computed statistics for longitude:
mean: [[26.761473]], variance: [[219.1066]]
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ pickup_longitude    │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropoff_longitude   │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pickup_latitude     │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │   

/home/jupyter/asl-ml-immersion/asl_core/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 1: saving model to ./taxifare-model/checkpoint.keras
20/20 - 13s - 656ms/step - loss: 170.9923 - root_mean_squared_error: 13.0764 - val_loss: 198.3106 - val_root_mean_squared_error: 14.0823
Saved artifact at './taxifare-model/savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 1), dtype=tf.float32, name='pickup_longitude'), TensorSpec(shape=(None, 1), dtype=tf.float32, name='pickup_latitude'), TensorSpec(shape=(None, 1), dtype=tf.float32, name='dropoff_longitude'), TensorSpec(shape=(None, 1), dtype=tf.float32, name='dropoff_latitude')]
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  139961920245008: TensorSpec(shape=(1, 1), dtype=tf.float32, name=None)
  139961920245392: TensorSpec(shape=(1, 1), dtype=tf.float32, name=None)
  139961920244816: TensorSpec(shape=(1, 1), dtype=tf.float32, name=None)
  139961920245200: TensorSpec(shape=(1, 1), dtype=tf.float32, name=None)


## Run your training package on Vertex AI


Once the code works in standalone mode locally, you can run it on the Cloud using Vertex AI.

In Vertex AI, ou can provide training code to Vertex AI in one of the following forms:

- **A Python training application to use with a prebuilt container**. Create a [Python source distribution](https://packaging.python.org/en/latest/overview/#python-source-distributions) with code that trains an ML model and exports it to Cloud Storage. This training application can use any of the dependencies included in the prebuilt container that you plan to use it with. Use this option if one of the Vertex AI prebuilt containers for training includes all the dependencies that you need for training.

- **A custom container image**. Create a Docker container image with code that trains an ML model and exports it to Cloud Storage. Include any dependencies required by your code in the container image.


### Method 1: Prebuilt Container
First, let's run a cloud training using a prebuild containers.

In order to do so, we need to package our code as a source distribution. For this, we can use `setuptools`. 

In [17]:
%%writefile taxifare/setup.py
"""Using `setuptools` to create a source distribution."""

from setuptools import find_packages, setup

setup(
    name="taxifare_trainer",
    version="0.1",
    packages=find_packages(),
    include_package_data=True,
    description="Taxifare model training application.",
)

Writing taxifare/setup.py


In [18]:
%%bash
cd taxifare
python ./setup.py sdist --formats=gztar
cd ..

running sdist
running egg_info
creating taxifare_trainer.egg-info
writing taxifare_trainer.egg-info/PKG-INFO
writing dependency_links to taxifare_trainer.egg-info/dependency_links.txt
writing top-level names to taxifare_trainer.egg-info/top_level.txt
writing manifest file 'taxifare_trainer.egg-info/SOURCES.txt'
reading manifest file 'taxifare_trainer.egg-info/SOURCES.txt'
writing manifest file 'taxifare_trainer.egg-info/SOURCES.txt'


running check
creating taxifare_trainer-0.1
creating taxifare_trainer-0.1/taxifare_trainer.egg-info
creating taxifare_trainer-0.1/trainer
copying files to taxifare_trainer-0.1...
copying setup.py -> taxifare_trainer-0.1
copying taxifare_trainer.egg-info/PKG-INFO -> taxifare_trainer-0.1/taxifare_trainer.egg-info
copying taxifare_trainer.egg-info/SOURCES.txt -> taxifare_trainer-0.1/taxifare_trainer.egg-info
copying taxifare_trainer.egg-info/dependency_links.txt -> taxifare_trainer-0.1/taxifare_trainer.egg-info
copying taxifare_trainer.egg-info/top_level.txt -> taxifare_trainer-0.1/taxifare_trainer.egg-info
copying trainer/__init__.py -> taxifare_trainer-0.1/trainer
copying trainer/model.py -> taxifare_trainer-0.1/trainer
copying trainer/task.py -> taxifare_trainer-0.1/trainer
copying taxifare_trainer.egg-info/SOURCES.txt -> taxifare_trainer-0.1/taxifare_trainer.egg-info
Writing taxifare_trainer-0.1/setup.cfg
creating dist
Creating tar archive
removing 'taxifare_trainer-0.1' (and everythi

We will store our package in the Cloud Storage bucket.

In [19]:
%%bash
gsutil cp taxifare/dist/taxifare_trainer-0.1.tar.gz gs://${BUCKET}/taxifare/

Copying file://taxifare/dist/taxifare_trainer-0.1.tar.gz [Content-Type=application/x-tar]...
/ [1 files][  3.5 KiB/  3.5 KiB]                                                
Operation completed over 1 objects/3.5 KiB.                                      


#### Submit Custom Job using the Python SDK

To submit this source distribution to the Cloud we use [CustomJob](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.CustomJob#google_cloud_aiplatform_CustomJob_get) module under Vertex AI Python SDK, and specify some parameters for Vertex AI Training service under [worker_pool_spec](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform_v1.types.WorkerPoolSpec), which includes:
- [`machine_spec`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform_v1.types.MachineSpec): Specification of a single machine where we run training. Here we can speficy `machine_type`, as well as the [accelerator specifications](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform_v1.types.AcceleratorType).
- `python_package_spec`: Here we provide specification about our Python package runtime, including the prebuilt container URL (`executor_image_uri`), our Python package path (`package_uris`), as well as the custom arguments (`args`) we pass to our training application can be defined and provided here.

Because this is on the entire dataset, it will take a while. You can monitor the job from the GCP console in the Vertex AI Training section.

**Lab Task #2**: Train your model using cloud infrastructure via Google Cloud Vertex AI Training Service
Fill in the TODOs in the code below to submit your job for training on Vertex AI. 

In [21]:
BATCH_SIZE = 64
NUM_EXAMPLES_TO_TRAIN_ON = 500000
NUM_EVALS = 100
NBUCKETS = 10
LR = 0.001
NNSIZE = "32 8"

base_path = f"gs://{BUCKET}/taxifare"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

args = [
    "--eval_data_path",
    f"{base_path}/data/taxi-valid*",
    "--output_dir",
    f"{base_path}/trained_model_{timestamp}",
    "--train_data_path",
    f"{base_path}/data/taxi-train*",
    "--batch_size",
    f"{BATCH_SIZE}",
    "--num_examples_to_train_on",
    f"{NUM_EXAMPLES_TO_TRAIN_ON}",
    "--num_evals",
    f"{NUM_EVALS}",
    "--nbuckets",
    f"{NBUCKETS}",
    "--lr",
    f"{LR}",
    "--nnsize",
    f"{NNSIZE}",
]

worker_pool_specs = [
    {
        "machine_spec": {
            "machine_type": "n1-standard-4",
            "accelerator_type": None,
            "accelerator_count": None,
        },
        "replica_count": 1,
        "python_package_spec": {
            "executor_image_uri": "us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-17.py310:latest",
            "package_uris": [f"{base_path}/taxifare_trainer-0.1.tar.gz"],
            "python_module": "trainer.task",
            "args": args,
        },
    }
]

training_job = aiplatform.CustomJob(
     display_name=f"taxifare_{timestamp}",
    worker_pool_specs=worker_pool_specs,
    staging_bucket=f"{base_path}/staging",
)

training_job.submit()

Creating CustomJob
CustomJob created. Resource name: projects/998865242242/locations/us-central1/customJobs/5852206628031430656
To use this CustomJob in another session:
custom_job = aiplatform.CustomJob.get('projects/998865242242/locations/us-central1/customJobs/5852206628031430656')
View Custom Job:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/5852206628031430656?project=998865242242


**Note:** you can run a custom training via bash command, using [`gcloud ai custom-jobs create`](https://cloud.google.com/sdk/gcloud/reference/ai/custom-jobs/create).

Equivalent bash command for reference:
```bash
TIMESTAMP=$(date -u +%Y%m%d_%H%M%S)
OUTDIR=gs://${BUCKET}/taxifare/trained_model_$TIMESTAMP
JOB_NAME=taxifare_$TIMESTAMP
echo ${OUTDIR} ${REGION} ${JOB_NAME}

PYTHON_PACKAGE_URIS=gs://${BUCKET}/taxifare/taxifare_trainer-0.1.tar.gz
MACHINE_TYPE=n1-standard-4
REPLICA_COUNT=1
PYTHON_PACKAGE_EXECUTOR_IMAGE_URI="us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-17.py310:latest"
PYTHON_MODULE=trainer.task

# Model and training hyperparameters
BATCH_SIZE=64
NUM_EXAMPLES_TO_TRAIN_ON=500000
NUM_EVALS=100
NBUCKETS=10
LR=0.001
NNSIZE="32 8"

# GCS paths
GCS_PROJECT_PATH=gs://$BUCKET/taxifare
DATA_PATH=$GCS_PROJECT_PATH/data
TRAIN_DATA_PATH=$DATA_PATH/taxi-train*
EVAL_DATA_PATH=$DATA_PATH/taxi-valid*

WORKER_POOL_SPEC="machine-type=$MACHINE_TYPE,\
replica-count=$REPLICA_COUNT,\
executor-image-uri=$PYTHON_PACKAGE_EXECUTOR_IMAGE_URI,\
python-module=$PYTHON_MODULE"

ARGS="--eval_data_path=$EVAL_DATA_PATH,\
--output_dir=$OUTDIR,\
--train_data_path=$TRAIN_DATA_PATH,\
--batch_size=$BATCH_SIZE,\
--num_examples_to_train_on=$NUM_EXAMPLES_TO_TRAIN_ON,\
--num_evals=$NUM_EVALS,\
--nbuckets=$NBUCKETS,\
--lr=$LR,\
--nnsize=$NNSIZE"

gcloud ai custom-jobs create \
  --region=${REGION} \
  --display-name=$JOB_NAME \
  --python-package-uris=$PYTHON_PACKAGE_URIS \
  --worker-pool-spec=$WORKER_POOL_SPEC \
  --args="$ARGS"
```

#### Open TensorBoard
Since we specified the `TensorBoard` callback in `model.fit`, the training application saved the intermediate logs for TensorBoard dashboard.

While we can [host TensorBoard in Vertex AI Experiments](https://cloud.google.com/vertex-ai/docs/experiments/tensorboard-introduction), here, let's simply open from this notebook.

You might not see any data for a bit until the job begins. Check the job status on the console, and then return here to click the refresh button in the top right to update TensorBoard.

In [ ]:
%tensorboard --logdir {base_path}/trained_model_{timestamp} --port 8000

### Method 2: Using a custom container

Vertex AI Training also supports training in custom containers, allowing users to bring their own Docker containers with any pre-installed ML framework or algorithm to run on Vertex AI Training. 

In this last section, we'll see how to submit a Cloud training job using a customized Docker image. 

Containerizing our `./taxifare/trainer` package involves 3 steps:

* Writing a Dockerfile in `./taxifare`
* Building the Docker image
* Pushing it to the Google Cloud Artifact Registry in our GCP project

The `Dockerfile` specifies
1. How the container needs to be provisioned so that all the dependencies in our code are satisfied
2. Where to copy our trainer Package in the container
3. What command to run when the container is run (the `ENTRYPOINT` line)

**Lab Task #3**: Running your training package using Docker containers.
Fill in the TODOs in the code below for Dockerfile

In [22]:
%%writefile ./taxifare/Dockerfile
FROM us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-17.py310:latest
# TODO 3

COPY . /code

WORKDIR /code

ENTRYPOINT ["python3", "-m", "trainer.task"]

Overwriting ./taxifare/Dockerfile


In [23]:
ARTIFACT_REGISTRY_DIR = "asl-artifact-repo"
IMAGE_NAME = "taxifare_training_container"
IMAGE_URI = f"us-docker.pkg.dev/{PROJECT}/{ARTIFACT_REGISTRY_DIR}/{IMAGE_NAME}"

os.environ["IMAGE_URI"] = IMAGE_URI

In [24]:
%%bash 

PROJECT_DIR=$(cd ./taxifare && pwd)
DOCKERFILE=$PROJECT_DIR/Dockerfile

# Authorize docker command for Artifact Registry
gcloud auth print-access-token | docker login -u oauth2accesstoken --password-stdin https://us-docker.pkg.dev

docker build $PROJECT_DIR -f $DOCKERFILE -t $IMAGE_URI

docker push $IMAGE_URI

WARNING! Your password will be stored unencrypted in /home/jupyter/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores



Login Succeeded


#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 190B done
#1 DONE 0.0s

#2 [internal] load metadata for us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-17.py310:latest
#2 DONE 0.2s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [internal] load build context
#4 transferring context: 111.20kB done
#4 DONE 0.0s

#5 [1/3] FROM us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-17.py310:latest@sha256:532ce4f03fa6a459bbdb23ee70102cd2bf0d4b626d366e060e9431206b0536c6
#5 resolve us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-17.py310:latest@sha256:532ce4f03fa6a459bbdb23ee70102cd2bf0d4b626d366e060e9431206b0536c6 0.0s done
#5 sha256:532ce4f03fa6a459bbdb23ee70102cd2bf0d4b626d366e060e9431206b0536c6 6.37kB / 6.37kB done
#5 sha256:f406ab404c7b5535f2ed39d0a35a34764accca4358d2f306d3f29aeda79c4a50 0B / 3.27kB 0.1s
#5 sha256:92899d74e2af66b90fd7edaed329ae5d384190adb2aa0036919f2be1ee0ee9a8

Using default tag: latest
The push refers to repository [us-docker.pkg.dev/qwiklabs-asl-00-8534302ca246/asl-artifact-repo/taxifare_training_container]
5f70bf18a086: Preparing
da74a431648c: Preparing
5f70bf18a086: Preparing
4529b7832e92: Preparing
df191efa0b07: Preparing
1807154ca784: Preparing
4dabb6c7ee83: Preparing
5f70bf18a086: Preparing
c404eff00392: Preparing
e579ed2157f0: Preparing
5f70bf18a086: Preparing
4a9ac05e89b4: Preparing
5f70bf18a086: Preparing
ae5069f08356: Preparing
67774152c98d: Preparing
1643ae49785c: Preparing
645cfb91788b: Preparing
a6e03b86290c: Preparing
5f70bf18a086: Preparing
25778a418cd6: Preparing
5e04d513d718: Preparing
5f70bf18a086: Preparing
b73d54578506: Preparing
bafae0f45f74: Preparing
5a349197fb12: Preparing
a2ea8902b696: Preparing
79de1f334f98: Preparing
26af3f41f81b: Preparing
27ab02f6b03c: Preparing
fd640bf432e0: Preparing
8d6b7eb76b62: Preparing
4a9ac05e89b4: Waiting
ae5069f08356: Waiting
67774152c98d: Waiting
1643ae49785c: Waiting
645cfb91788b: Wai

#### Submit Custom Job using the Python SDK

As we did above, let's define the worker_pool_spec using Vertex AI Python SDK and run the cloud training.

The definition is almost the same, but please note that here we specify `container_spec` instead of `python_package_spec`, which simply includes our custom container image path.

In [25]:
BATCH_SIZE = 64
NUM_EXAMPLES_TO_TRAIN_ON = 500000
NUM_EVALS = 100
NBUCKETS = 10
LR = 0.001
NNSIZE = "32 8"

base_path = f"gs://{BUCKET}/taxifare"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

args = [
    "--eval_data_path",
    f"{base_path}/data/taxi-valid*",
    "--output_dir",
    f"{base_path}/trained_model_{timestamp}",
    "--train_data_path",
    f"{base_path}/data/taxi-train*",
    "--batch_size",
    f"{BATCH_SIZE}",
    "--num_examples_to_train_on",
    f"{NUM_EXAMPLES_TO_TRAIN_ON}",
    "--num_evals",
    f"{NUM_EVALS}",
    "--nbuckets",
    f"{NBUCKETS}",
    "--lr",
    f"{LR}",
    "--nnsize",
    f"{NNSIZE}",
]

worker_pool_specs = [
    {
        "machine_spec": {
            "machine_type": "n1-standard-4",
            "accelerator_type": None,
            "accelerator_count": None,
        },
        "replica_count": 1,
        # Now we specify container_spec, instead of python_package_spec
        "container_spec": {
            "image_uri": IMAGE_URI,
            "args": args,
        },
    }
]

training_job = aiplatform.CustomJob(
    display_name=f"taxifare_container_{timestamp}",
    worker_pool_specs=worker_pool_specs,
    staging_bucket=f"{base_path}/staging",
)

training_job.submit()

Creating CustomJob
CustomJob created. Resource name: projects/998865242242/locations/us-central1/customJobs/7924284669086924800
To use this CustomJob in another session:
custom_job = aiplatform.CustomJob.get('projects/998865242242/locations/us-central1/customJobs/7924284669086924800')
View Custom Job:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/7924284669086924800?project=998865242242


**Note:** Here is the equivalent bash command to run container training for reference.

```bash
# Output directory and jobID
OUTDIR=gs://${BUCKET}/taxifare/trained_model_$TIMESTAMP
JOB_NAME=taxifare_container_$TIMESTAMP
echo ${OUTDIR} ${REGION} ${JOB_NAME}

# Model and training hyperparameters
BATCH_SIZE=64
NUM_EXAMPLES_TO_TRAIN_ON=500000
NUM_EVALS=100
NBUCKETS=10
LR=0.001
NNSIZE="32 8"

# Vertex AI machines to use for training
MACHINE_TYPE=n1-standard-4
REPLICA_COUNT=1

# GCS paths.
GCS_PROJECT_PATH=gs://$BUCKET/taxifare
DATA_PATH=$GCS_PROJECT_PATH/data
TRAIN_DATA_PATH=$DATA_PATH/taxi-train*
EVAL_DATA_PATH=$DATA_PATH/taxi-valid*

WORKER_POOL_SPEC="machine-type=$MACHINE_TYPE,\
replica-count=$REPLICA_COUNT,\
container-image-uri=$IMAGE_URI"

ARGS="--eval_data_path=$EVAL_DATA_PATH,\
--output_dir=$OUTDIR,\
--train_data_path=$TRAIN_DATA_PATH,\
--batch_size=$BATCH_SIZE,\
--num_examples_to_train_on=$NUM_EXAMPLES_TO_TRAIN_ON,\
--num_evals=$NUM_EVALS,\
--nbuckets=$NBUCKETS,\
--lr=$LR,\
--nnsize=$NNSIZE"

gcloud ai custom-jobs create \
  --region=$REGION \
  --display-name=$JOB_NAME \
  --worker-pool-spec=$WORKER_POOL_SPEC \
  --args="$ARGS"
```

#### Open TensorBoard

Let's check TensorBoard once more. A different port number will be used this time, as `8000` is occupied by another TensorBoard instance above.

In [ ]:
%tensorboard --logdir {base_path}/trained_model_{timestamp} --port 8001

Copyright 2025 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License